In [39]:
import numpy as np
from tqdm import tqdm
from games.kuhn import KuhnPoker
from agents.counterfactualregret_t import CounterFactualRegret
from agents.ismcts import ISMCTS
from agents.minimax import MiniMax
from agents.agent_random import RandomAgent

In [40]:
g = KuhnPoker(initial_player=0)

In [52]:
my_agents = {
    g.agents[0]: CounterFactualRegret(game=g, agent=g.agents[0]),
    #g.agents[0]: RandomAgent(game=g, agent=g.agents[1]),
    #g.agents[1]: ISMCTS(game=g, agent=g.agents[1], simulations=200, rollouts=10, rollout_iterations=100),
    g.agents[1]: CounterFactualRegret(game=g, agent=g.agents[1])
}

In [53]:
g.reset()
while not g.done():
    g.render()
    print(f"Agent {g.agent_selection}")
    action = my_agents[g.agent_selection].action()
    print(f"Action {action} - move {g.action_move(action)}")
    g.step(action)
g.render()
for agent in g.agents:
    print(f"Reward {agent} = {g.reward(agent)}")

agent_0 Q 
agent_1 J 
Agent agent_0
Node 1 does not exist. Playing random.
Action 0 - move p
agent_0 Q p
agent_1 J p
Agent agent_1
Node 0p does not exist. Playing random.
Action 1 - move b
agent_0 Q pb
agent_1 J pb
Agent agent_0
Node 1pb does not exist. Playing random.
Action 0 - move p
agent_0 Q pbp
agent_1 J pbp
Reward agent_0 = -1
Reward agent_1 = 1


In [ ]:
train_iterations = 1000

for agent in g.agents:
    print('Training agent ' + agent)
    if hasattr(my_agents[agent], 'train'):
        my_agents[agent].train(train_iterations)
        print(dict(map(lambda n: (n, np.round(my_agents[agent].node_dict[n].policy(), 3)), my_agents[agent].node_dict.keys())))

Training agent agent_0


100%|██████████| 1000/1000 [00:03<00:00, 252.83it/s]


{'2': array([0.327, 0.673]), '1p': array([0.998, 0.002]), '2pb': array([0., 1.]), '1b': array([0.652, 0.348]), '1': array([0.995, 0.005]), '2p': array([0., 1.]), '1pb': array([0.381, 0.619]), '2b': array([0., 1.]), '0p': array([0.647, 0.353]), '0b': array([1., 0.]), '0': array([0.799, 0.201]), '0pb': array([1., 0.])}
Training agent agent_1


100%|██████████| 1000/1000 [00:04<00:00, 233.93it/s]

{'0': array([0.806, 0.194]), '2p': array([0., 1.]), '0pb': array([1., 0.]), '2b': array([0., 1.]), '2': array([0.304, 0.696]), '1p': array([0.998, 0.002]), '2pb': array([0., 1.]), '1b': array([0.671, 0.329]), '1': array([0.99, 0.01]), '1pb': array([0.388, 0.612]), '0p': array([0.659, 0.341]), '0b': array([1., 0.])}


In [75]:
cum_rewards = dict(map(lambda agent: (agent, 0.), g.agents))
niter = 100
for _ in tqdm(range(niter)):
    g.reset()
    #g.render()
    turn = 0
    while not g.done():
        #print('Turn: ', turn)
        #print('\tPlayer: ', g.agent_selection)
        #print('\tObservation: ', g.observe(g.agent_selection))
        a = my_agents[g.agent_selection].action()
        #print('\tAction: ', g._moves[a])
        g.step(action=a)
        turn += 1
    #print('Rewards: ', g.rewards)
    for agent in g.agents:
        cum_rewards[agent] += g.rewards[agent]
print('Average rewards:', dict(map(lambda agent: (agent, cum_rewards[agent]/niter), g.agents)))


100%|██████████| 100/100 [00:00<00:00, 3687.49it/s]

Average rewards: {'agent_0': -0.01, 'agent_1': 0.01}
